# Prep 6: Variational Autoencoder (VAE)

Source: [Official PyTorch VAE example](https://github.com/pytorch/examples/tree/main/vae)

## Learning purpose

Build a mental model of how a VAE compresses, reconstructs, and samples images.

## Why this matters

A VAE is a model that learns to compress an image into a small hidden code, then rebuild the image from that code. The hidden code lives in a **latent space**, which means a space of learned numbers that represent useful image features rather than raw pixels. This notebook makes that idea visible by saving reconstruction grids, sample grids, and possibly interpolation grids. Later, diffusion models will reuse the same broad idea: generate images by working with learned representations instead of only thinking in raw pixels.

## Core idea

For MNIST, each digit image has `28 × 28 = 784` pixel values. A basic autoencoder learns this pattern:

```text
image pixels → encoder → small code → decoder → rebuilt image
```

The PyTorch example compresses each image into a 20-number latent representation, then decodes that representation back into 784 pixel values. This bottleneck matters because the model cannot simply copy every pixel; it has to learn useful digit structure such as loops, strokes, and slants.

## What makes it variational?

A regular autoencoder can map an image to one exact code. A VAE maps an image to a small probability cloud:

```text
image → mean + spread → sampled latent code → rebuilt image
```

In the official code, the encoder returns `mu` and `logvar`. `mu` is the center of the cloud, `logvar` describes its spread, and `z` is a sampled point from that cloud. This organized sampling step is what lets the decoder create new digit-like images from random latent codes.

## Training loop in plain language

For each batch of images, the VAE:

1. reads real digit images,
2. encodes each image into `mu` and `logvar`,
3. samples a latent code `z`,
4. decodes `z` into a reconstructed image,
5. measures reconstruction loss: how close the rebuilt image is to the original,
6. measures KL loss: how well the latent clouds stay organized for sampling,
7. updates the model weights to reduce the combined loss.

```text
total loss = reconstruction loss + KL loss
```

## What the saved grids should show

- **Reconstruction grid:** original digits beside rebuilt digits. This checks whether the latent code keeps enough information to preserve digit identity.
- **Sample grid:** random latent codes decoded into images. This checks whether random points in latent space become digit-like outputs.
- **Interpolation grid:** a smooth walk between two latent codes. This checks whether nearby latent points create gradual visual changes.

Blurry but recognizable digits are a useful early result. The goal is not perfect handwriting; the goal is to make compression, sampling, and latent-space structure visible.


## Notebook stage 1: setup

This cell prepares the notebook before any model training happens. It imports the PyTorch tools used later, finds the project root by looking for `pyproject.toml`, sets a fixed random seed, chooses `cuda` when a GPU is available, and creates `outputs/prep/vae/` for generated image grids.

- **Project root**: the repository folder, used so paths do not depend on where Jupyter started.
- **Device**: the hardware target for tensor math, usually `cuda` for GPU or `cpu` otherwise.
- **Seed**: a starting value for random number generation, used to make repeated runs easier to compare.
- **Output directory**: the folder where reconstruction, sample, and interpolation images will be saved.


In [12]:
# Setup: imports, project paths, seed, and device

from pathlib import Path

import torch
from torch import nn, optim
from torch.nn import functional as F
from torchvision import datasets, transforms


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


torch.manual_seed(1)

project_root = find_project_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = project_root / "outputs" / "prep" / "vae"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Using device: {device}")
print(f"Saving VAE artifacts to: {output_dir}")

Project root: C:\Users\giloz\dev\visual-genai-lab
Using device: cuda
Saving VAE artifacts to: C:\Users\giloz\dev\visual-genai-lab\outputs\prep\vae


## Notebook stage 2: MNIST data loaders

This cell downloads MNIST and wraps it in `DataLoader` objects. MNIST is a dataset of small handwritten digit images; each image is `28 × 28` grayscale pixels. The VAE will learn from batches of these images, then we will inspect whether it can reconstruct digits and sample new ones.

- **Transform**: `transforms.ToTensor()` converts each image into a PyTorch tensor, changes pixel values from `0–255` into `0.0–1.0`, and gives each image shape `[1, 28, 28]`.
- **Pinned memory**: `pin_memory=True` can speed up CPU-to-GPU transfer, but it does not change the image data and does not move data to the GPU by itself.


In [13]:
# MNIST data: download digits and create training/test loaders
# Source: https://github.com/pytorch/examples/tree/main/vae

batch_size = 128
data_dir = project_root / "data"

transform = transforms.ToTensor()

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root=data_dir,
        train=True,
        download=True,
        transform=transform,
    ),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root=data_dir,
        train=False,
        download=True,
        transform=transform,
    ),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

images, labels = next(iter(train_loader))

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"First labels: {labels[:10].tolist()}")

Training batches: 469
Test batches: 79
Image batch shape: torch.Size([128, 1, 28, 28])
Label batch shape: torch.Size([128])
First labels: [4, 8, 8, 6, 7, 1, 0, 7, 1, 8]


## Notebook stage 3: high-level VAE math

Before reading the architecture code, it helps to name the math operations the model is trying to perform. The symbols below are a map, not a proof. The code cell after this section implements the same flow with PyTorch layers.

1. **Start with an image batch**

   `x` is a batch of MNIST images with shape `[batch_size, 1, 28, 28]`. Each image contains pixel values between `0.0` and `1.0`.

2. **Flatten each image**

   ```text
   [1, 28, 28] → [784]
   ```

   The model changes each image grid into one long list of 784 pixel numbers so linear layers can process it.

3. **Encode pixels into a hidden representation**

   ```text
   h = ReLU(W₁x + b₁)
   ```

   `W` means learned weights, `b` means learned bias values, and `ReLU` keeps positive values while turning negative values into zero. This produces a smaller learned summary of the image.

4. **Predict a latent distribution**

   ```text
   μ = Wμh + bμ
   log(σ²) = Wσh + bσ
   ```

   The encoder does not produce one exact latent code. It produces `μ` (`mu`), the center of a 20-number latent cloud, and `logvar`, the log of the variance, which describes the cloud’s spread.

5. **Sample a latent code**

   ```text
   z = μ + σ × ε, where ε is random noise from a normal distribution
   ```

   This gives one sampled 20-number code `z` for each image. The model learns an organized latent space instead of memorizing one fixed code per image.

6. **Decode the latent code back into pixels**

   ```text
   reconstructed_pixels = sigmoid(W₄ ReLU(W₃z + b₃) + b₄)
   ```

   The decoder maps the 20-number latent code back to 784 pixel values. `sigmoid` squeezes the output into the `0.0` to `1.0` range, matching the MNIST tensor values.

7. **Train with two goals**

   ```text
   total loss = reconstruction loss + KL loss
   ```

   Reconstruction loss asks: “does the rebuilt image match the original image?” KL loss asks: “is the latent space organized enough that random samples can decode into digit-like images?”

### Why the KL term keeps latent space sampleable

A **latent cloud** is the fuzzy region of possible `z` values around one image’s `mu`. The encoder predicts the cloud’s center with `mu` and its spread with `logvar`; `reparameterize()` then picks one point from that cloud.

A **sampleable latent space** means that random latent codes can be decoded into useful digit-like images. This matters because generation later starts with random codes such as `torch.randn(64, latent_dim)`, not with real MNIST images.

The KL term gently pulls each image’s latent cloud toward a **standard normal** shape: centered near `0` with spread near `1`. If the useful latent codes became tiny scattered islands with large empty gaps between them, random samples would often land in places the decoder never learned to handle. Keeping the clouds near the same simple normal region makes random sampling much more likely to produce recognizable outputs.

This is a balance, not a rule that all digits must become identical. Reconstruction loss pushes the model to keep enough information to rebuild the input image; KL loss pushes the latent space to stay organized enough for sampling.

### Code-to-math naming map

| Math idea | Code name | What it means |
| --- | --- | --- |
| Input image batch `x` | `images`, then `x` inside `forward()` | A batch of digit images shaped `[batch_size, 1, 28, 28]`. |
| Flattened pixels | `x.view(-1, 784)` | Each image grid becomes one list of 784 pixel numbers. |
| Hidden representation `h` | `h1` | The post-ReLU output of `fc1`; a learned 400-number summary of each image. |
| Mean `μ` | `mu`, from `fc21(h1)` | The center of each image’s 20-number latent cloud. |
| Log variance `log(σ²)` | `logvar`, from `fc22(h1)` | The spread information for that latent cloud, stored in log form. |
| Sampled latent code `z` | `z` | One sampled 20-number code made from `mu`, `logvar`, and random noise. |
| Decoder hidden representation | `h3` | The post-ReLU output of `fc3`; a learned expansion from latent code back toward pixels. |
| Reconstructed pixels | `recon_batch` | The decoder output shaped `[batch_size, 784]`, later viewable as 28×28 images. |


In [14]:
# VAE model architecture
# Source: https://github.com/pytorch/examples/tree/main/vae

# The latent dimension is the number of hidden numbers used to represent each digit.
# In the math cell, this is the size of mu, logvar, and z for one image.
latent_dim = 20


class VAE(nn.Module):
    def __init__(self, latent_dim: int = 20) -> None:
        super().__init__()
        # Encoder: flattened pixels x -> hidden representation h1.
        # Math map: h = ReLU(W₁x + b₁).
        self.fc1 = nn.Linear(784, 400)

        # Two encoder heads read the same h1 features.
        # fc21 predicts mu: the center of the latent cloud.
        # fc22 predicts logvar: the log-spread of the latent cloud.
        self.fc21 = nn.Linear(400, latent_dim)
        self.fc22 = nn.Linear(400, latent_dim)

        # Decoder: sampled latent code z -> hidden representation h3 -> pixels.
        self.fc3 = nn.Linear(latent_dim, 400)
        self.fc4 = nn.Linear(400, 784)

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # h1 means hidden representation 1: the post-ReLU output of fc1.
        # Shape map: [batch_size, 784] -> [batch_size, 400].
        h1 = F.relu(self.fc1(x))

        # Both heads use the same h1 but learn different distribution parameters.
        # Shape map for each head: [batch_size, 400] -> [batch_size, latent_dim].
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        # logvar stores log(variance). Multiplying by 0.5 converts it toward log(std),
        # and exp(...) turns it back into std, the standard deviation σ.
        std = torch.exp(0.5 * logvar)

        # eps is the random normal noise ε in z = μ + σ × ε.
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # h3 means hidden representation 3: the decoder hidden layer after sampling z.
        h3 = F.relu(self.fc3(z))

        # Sigmoid squeezes reconstructed pixels into the 0.0–1.0 range.
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Flatten image grids, then encode them into distribution parameters.
        mu, logvar = self.encode(x.view(-1, 784))

        # Sample one latent code z from each image's learned latent cloud.
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


model = VAE(latent_dim=latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

images = images.to(device)
recon_batch, mu, logvar = model(images)

print(f"Input image batch shape: {images.shape}")
print(f"Reconstruction batch shape: {recon_batch.shape}")
print(f"mu shape: {mu.shape}")
print(f"logvar shape: {logvar.shape}")

Input image batch shape: torch.Size([128, 1, 28, 28])
Reconstruction batch shape: torch.Size([128, 784])
mu shape: torch.Size([128, 20])
logvar shape: torch.Size([128, 20])


## Notebook stage 4: VAE loss

The VAE loss has two parts. Reconstruction loss asks whether the decoded pixels match the original image pixels. KL loss asks whether the encoder’s latent clouds stay close to a simple normal distribution, which keeps the latent space usable for sampling.

The model trains by minimizing the sum of both terms.

In [15]:
# VAE loss: reconstruction quality + latent-space organization
# Source: https://github.com/pytorch/examples/tree/main/vae


def loss_function(
    recon_x: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
) -> torch.Tensor:
    # Reconstruction term:
    # Compare decoded pixels [batch_size, 784] with original flattened pixels [batch_size, 784].
    reconstruction_loss = F.binary_cross_entropy(
        recon_x,
        x.view(-1, 784),
        reduction="sum",
    )

    # KL term:
    # Encourage each latent cloud, described by mu and logvar, to stay near a standard normal cloud.
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return reconstruction_loss + kl_loss